In [1]:
from dotenv import load_dotenv
import os
import requests

load_dotenv()

token = os.getenv("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}
response = requests.get(
    "https://api.github.com/user",
    headers=headers
)

In [2]:
data = response.json()

In [3]:
data

{'login': 'kichu3000',
 'id': 159616764,
 'node_id': 'U_kgDOCYOO_A',
 'avatar_url': 'https://avatars.githubusercontent.com/u/159616764?v=4',
 'gravatar_id': '',
 'url': 'https://api.github.com/users/kichu3000',
 'html_url': 'https://github.com/kichu3000',
 'followers_url': 'https://api.github.com/users/kichu3000/followers',
 'following_url': 'https://api.github.com/users/kichu3000/following{/other_user}',
 'gists_url': 'https://api.github.com/users/kichu3000/gists{/gist_id}',
 'starred_url': 'https://api.github.com/users/kichu3000/starred{/owner}{/repo}',
 'subscriptions_url': 'https://api.github.com/users/kichu3000/subscriptions',
 'organizations_url': 'https://api.github.com/users/kichu3000/orgs',
 'repos_url': 'https://api.github.com/users/kichu3000/repos',
 'events_url': 'https://api.github.com/users/kichu3000/events{/privacy}',
 'received_events_url': 'https://api.github.com/users/kichu3000/received_events',
 'type': 'User',
 'user_view_type': 'public',
 'site_admin': False,
 'nam

In [4]:
import requests

url = "https://api.github.com/repos/facebook/react"

response = requests.get(url, headers=headers)

print("Status:", response.status_code)

repo = response.json()

print(repo.keys())

Status: 200
dict_keys(['id', 'node_id', 'name', 'full_name', 'private', 'owner', 'html_url', 'description', 'fork', 'url', 'forks_url', 'keys_url', 'collaborators_url', 'teams_url', 'hooks_url', 'issue_events_url', 'events_url', 'assignees_url', 'branches_url', 'tags_url', 'blobs_url', 'git_tags_url', 'git_refs_url', 'trees_url', 'statuses_url', 'languages_url', 'stargazers_url', 'contributors_url', 'subscribers_url', 'subscription_url', 'commits_url', 'git_commits_url', 'comments_url', 'issue_comment_url', 'contents_url', 'compare_url', 'merges_url', 'archive_url', 'downloads_url', 'issues_url', 'pulls_url', 'milestones_url', 'notifications_url', 'labels_url', 'releases_url', 'deployments_url', 'created_at', 'updated_at', 'pushed_at', 'git_url', 'ssh_url', 'clone_url', 'svn_url', 'homepage', 'size', 'stargazers_count', 'watchers_count', 'language', 'has_issues', 'has_projects', 'has_downloads', 'has_wiki', 'has_pages', 'has_discussions', 'forks_count', 'mirror_url', 'archived', 'disab

In [6]:
import requests

url = "https://api.github.com/search/repositories"

params = {
    "q": "stars:>1000",
    "sort": "stars",
    "order": "desc",
    "per_page": 10
}

response = requests.get(url, headers=headers, params=params)

print("Status:", response.status_code)

data = response.json()

print("Total found:", data["total_count"])

for repo in data["items"]:
    print(repo["full_name"], repo["stargazers_count"])

Status: 200
Total found: 64908
codecrafters-io/build-your-own-x 548162
sindresorhus/awesome 507686
public-apis/public-apis 481526
freeCodeCamp/freeCodeCamp 455761
EbookFoundation/free-programming-books 397191
openclaw/openclaw 390067
donnemartin/system-design-primer 370713
nilbuild/developer-roadmap 367669
jwasham/coding-interview-university 361190
vinta/awesome-python 321640


In [7]:
import pandas as pd

df =pd.read_csv("github_repos_100k_raw.csv")

df.shape

C:\Users\Anandakrishnan VB\AppData\Local\Temp\ipykernel_11404\430513588.py:3: DtypeWarning: Columns (0: mirror_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df =pd.read_csv("github_repos_100k_raw.csv")


(53659, 83)

In [10]:
import requests
import pandas as pd
import time

url = "https://api.github.com/search/repositories"

languages = [
    "Python",
    "JavaScript",
    "Java",
    "C++",
    "C",
    "TypeScript",
    "Go",
    "C#",
    "Rust",
    "PHP",
    "Kotlin",
    "Swift",
    "Dart"
]

star_ranges = [
    "10..100",
    "101..500",
    "501..2000",
    "2001..10000",
    ">10000"
]

repos = []
seen = set(df["id"])

# Download repositories
for language in languages:

    for stars in star_ranges:

        query = f"language:{language} stars:{stars}"

        print("\nSearching:", query)

        for page in range(1, 11):

            params = {
                "q": query,
                "per_page": 100,
                "page": page
            }

            # Retry if rate limited
            while True:

                response = requests.get(
                    url,
                    headers=headers,
                    params=params
                )

                if response.status_code == 200:
                    break

                elif response.status_code == 403:
                    print("Rate limited. Waiting 60 seconds...")
                    time.sleep(60)

                else:
                    print("Error:", response.status_code)
                    print(response.json())
                    break

            if response.status_code != 200:
                break

            data = response.json()

            items = data.get("items", [])

            if not items:
                break

            # Remove duplicates
            for repo in items:

                if repo["id"] not in seen:
                    seen.add(repo["id"])
                    repos.append(repo)

            print(
                language,
                stars,
                "page",
                page,
                "total:",
                len(repos)
            )

            # Save progress every 1,000 repositories
            if len(repos) % 1000 == 0:

                df = pd.DataFrame(repos)

                df.to_csv(
                    "github_repos_progress.csv",
                    index=False
                )

                print("Progress saved:", len(repos))

            if len(repos) >= 100000:
                break

            print("Sleeping....")
            time.sleep(4)

        if len(repos) >= 100000:
            break

    if len(repos) >= 100000:
        break



repos = repos[:100000]

df = pd.DataFrame(repos)

df.to_csv(
    "github_repos_100k_raw.csv",
    index=False
)
print("---------------------------------------")
print("Download complete")
print("Repositories:", len(df))
print("Shape:", df.shape)
print("---------------------------------------")


Searching: language:Python stars:10..100
Python 10..100 page 1 total: 0
Progress saved: 0
Sleeping....
Python 10..100 page 2 total: 0
Progress saved: 0
Sleeping....
Python 10..100 page 3 total: 0
Progress saved: 0
Sleeping....
Python 10..100 page 4 total: 1
Sleeping....


KeyboardInterrupt: 

In [ ]:
import pandas as pd

df =pd.read_csv("github_repos_100k_raw.csv")

df.shape

C:\Users\Anandakrishnan VB\AppData\Local\Temp\ipykernel_11404\431238288.py:3: DtypeWarning: Columns (0: mirror_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df =pd.read_csv("github_repos_100k_raw.csv")


(53659, 83)

In [2]:
import requests
import pandas as pd
import time

# --------------------------------------------------
# SETTINGS
# --------------------------------------------------

url = "https://api.github.com/search/repositories"

languages = [
    "Python",
    "JavaScript",
    "Java",
    "C++",
    "C",
    "TypeScript",
    "Go",
    "C#",
    "Rust",
    "PHP",
    "Kotlin",
    "Swift",
    "Dart"
]

star_ranges = [
    "10..100",
    "101..500",
    "501..2000",
    "2001..10000",
    ">10000"
]

TARGET = 100000

# --------------------------------------------------
# LOAD EXISTING DATA
# --------------------------------------------------

df = pd.read_csv("github_repos_100k_raw.csv")

print("Existing repositories:", len(df))
print("Existing shape:", df.shape)

# IDs of repositories already collected
seen = set(df["id"])

# New repositories collected during this run
repos = []

print("---------------------------------------")
print("Starting collection...")
print("Already have:", len(df))
print("Still needed:", TARGET - len(df))
print("---------------------------------------")


# --------------------------------------------------
# DOWNLOAD NEW REPOSITORIES
# --------------------------------------------------

for language in languages:

    for stars in star_ranges:

        # Stop if target reached
        if len(df) + len(repos) >= TARGET:
            break

        query = f"language:{language} stars:{stars}"

        print("\nSearching:", query)

        for page in range(31, 40):

            # Stop if target reached
            if len(df) + len(repos) >= TARGET:
                break

            params = {
                "q": query,
                "per_page": 100,
                "page": page
            }

            # ------------------------------------------
            # REQUEST WITH RATE-LIMIT HANDLING
            # ------------------------------------------

            wait_time = 60

            while True:

                response = requests.get(
                    url,
                    headers=headers,
                    params=params
                )

                # Successful request
                if response.status_code == 200:
                    break

                # Rate limited
                elif response.status_code == 403 or response.status_code == 429:

                    retry_after = response.headers.get("Retry-After")

                    if retry_after:
                        wait = int(retry_after)
                    else:
                        wait = wait_time

                    print(
                        f"Rate limited. Waiting {wait} seconds..."
                    )

                    time.sleep(wait)

                    # Increase wait if it happens again
                    wait_time = min(wait_time * 2, 600)

                else:

                    print("Error:", response.status_code)

                    try:
                        print(response.json())
                    except:
                        print(response.text)

                    break

            # If request failed for another reason
            if response.status_code != 200:
                break

            # ------------------------------------------
            # GET RESULTS
            # ------------------------------------------

            data = response.json()

            items = data.get("items", [])

            if not items:
                print("No more results for this query.")
                break

            # ------------------------------------------
            # REMOVE DUPLICATES
            # ------------------------------------------

            new_count = 0

            for repo in items:

                repo_id = repo["id"]

                if repo_id not in seen:

                    seen.add(repo_id)
                    repos.append(repo)
                    new_count += 1

            total_now = len(df) + len(repos)

            print(
                language,
                stars,
                "page",
                page,
                "| New:",
                new_count,
                "| Total:",
                total_now
            )

            # ------------------------------------------
            # SAVE EVERY 1,000 NEW REPOSITORIES
            # ------------------------------------------

            if len(repos) > 0 and len(repos) % 1000 < new_count:

                df_progress = pd.concat(
                    [df, pd.DataFrame(repos)],
                    ignore_index=True
                )

                df_progress = df_progress.drop_duplicates(
                    subset="id"
                )

                df_progress.to_csv(
                    "github_repos_progress.csv",
                    index=False
                )

                print(
                    "Progress saved:",
                    len(df_progress),
                    "repositories"
                )

            # ------------------------------------------
            # TARGET REACHED
            # ------------------------------------------

            if total_now >= TARGET:
                break

            # ------------------------------------------
            # NORMAL DELAY
            # ------------------------------------------

            print("Sleeping 1 second...")
            time.sleep(1)

        if len(df) + len(repos) >= TARGET:
            break

    if len(df) + len(repos) >= TARGET:
        break


# --------------------------------------------------
# COMBINE OLD + NEW DATA
# --------------------------------------------------

df_final = pd.concat(
    [df, pd.DataFrame(repos)],
    ignore_index=True
)

# Remove duplicates
df_final = df_final.drop_duplicates(
    subset="id"
)

# Keep only first 100,000
df_final = df_final.iloc[:TARGET]


# --------------------------------------------------
# SAVE FINAL DATASET
# --------------------------------------------------

df_final.to_csv(
    "github_repos_100k_raw.csv",
    index=False
)


# --------------------------------------------------
# FINAL RESULT
# --------------------------------------------------

print("---------------------------------------")
print("Download complete")
print("Repositories:", len(df_final))
print("Shape:", df_final.shape)
print("---------------------------------------")

C:\Users\Anandakrishnan VB\AppData\Local\Temp\ipykernel_19608\2081716287.py:41: DtypeWarning: Columns (0: mirror_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("github_repos_100k_raw.csv")


Existing repositories: 53659
Existing shape: (53659, 83)
---------------------------------------
Starting collection...
Already have: 53659
Still needed: 46341
---------------------------------------

Searching: language:Python stars:10..100
Error: 422
{'message': 'Only the first 1000 search results are available', 'documentation_url': 'https://docs.github.com/v3/search/', 'status': '422'}

Searching: language:Python stars:101..500
Error: 422
{'message': 'Only the first 1000 search results are available', 'documentation_url': 'https://docs.github.com/v3/search/', 'status': '422'}

Searching: language:Python stars:501..2000
Error: 422
{'message': 'Only the first 1000 search results are available', 'documentation_url': 'https://docs.github.com/v3/search/', 'status': '422'}

Searching: language:Python stars:2001..10000
Error: 422
{'message': 'Only the first 1000 search results are available', 'documentation_url': 'https://docs.github.com/v3/search/', 'status': '422'}

Searching: language:

KeyboardInterrupt: 

In [3]:
df.shape

(53659, 83)

In [5]:
import requests
import pandas as pd
import time
from datetime import date, timedelta

# ============================================================
# SETTINGS
# ============================================================

url = "https://api.github.com/search/repositories"

TARGET = 100000

# How many results per API request
PER_PAGE = 100

# GitHub Search exposes only the first 1000 results
MAX_PAGES = 10

# Small delay between successful requests
REQUEST_DELAY = 1

# Delay after rate limiting
DEFAULT_WAIT = 60

# ============================================================
# LANGUAGES
# ============================================================

languages = [
    "Python",
    "JavaScript",
    "Java",
    "C++",
    "C",
    "TypeScript",
    "Go",
    "C#",
    "Rust",
    "PHP",
    "Kotlin",
    "Swift",
    "Dart"
]

# ============================================================
# STAR RANGES
# ============================================================

star_ranges = [
    "10..100",
    "101..500",
    "501..2000",
    "2001..10000",
    ">10000"
]

# ============================================================
# DATE RANGE
# ============================================================

START_YEAR = 2010
END_YEAR = 2026

# Split each year into 2 halves.
# If a query is still too large, the code can skip it
# and continue with other partitions.
#
# 2010-01-01..2010-06-30
# 2010-07-01..2010-12-31
# etc.

# ============================================================
# LOAD EXISTING DATA
# ============================================================

print("Loading existing dataset...")

df = pd.read_csv("github_repos_100k_raw.csv")

print("Existing repositories:", len(df))
print("Existing shape:", df.shape)

# ============================================================
# CHECK REQUIRED ID COLUMN
# ============================================================

if "id" not in df.columns:
    raise ValueError("The existing CSV does not contain an 'id' column.")

# Remove accidental duplicate IDs from old data
df = df.drop_duplicates(subset="id").reset_index(drop=True)

# Existing repository IDs
seen = set(df["id"].astype(int))

# New repositories collected during this run
repos = []

print("---------------------------------------")
print("Starting collection")
print("Existing:", len(df))
print("Still needed:", max(0, TARGET - len(df)))
print("---------------------------------------")


# ============================================================
# DATE PARTITION GENERATOR
# ============================================================

def generate_date_ranges(start_year, end_year):

    ranges = []

    for year in range(start_year, end_year + 1):

        # First half
        ranges.append(
            (
                f"{year}-01-01",
                f"{year}-06-30"
            )
        )

        # Second half
        ranges.append(
            (
                f"{year}-07-01",
                f"{year}-12-31"
            )
        )

    return ranges


date_ranges = generate_date_ranges(
    START_YEAR,
    END_YEAR
)

print("Date partitions:", len(date_ranges))


# ============================================================
# SAVE PROGRESS FUNCTION
# ============================================================

def save_progress():

    if len(repos) == 0:
        return

    new_df = pd.DataFrame(repos)

    combined = pd.concat(
        [df, new_df],
        ignore_index=True
    )

    combined = combined.drop_duplicates(
        subset="id"
    ).reset_index(drop=True)

    combined.to_csv(
        "github_repos_progress.csv",
        index=False
    )

    print()
    print(">>> PROGRESS SAVED")
    print("Repositories:", len(combined))
    print()


# ============================================================
# MAIN COLLECTION
# ============================================================

stop_collection = False

for language in languages:

    if stop_collection:
        break

    for stars in star_ranges:

        if stop_collection:
            break

        for start_date, end_date in date_ranges:

            if len(df) + len(repos) >= TARGET:
                stop_collection = True
                break

            query = (
                f"language:{language} "
                f"stars:{stars} "
                f"created:{start_date}..{end_date}"
            )

            print()
            print("=======================================")
            print("Searching:")
            print(query)
            print("=======================================")

            # ------------------------------------------------
            # Search pages 1-10
            # ------------------------------------------------

            for page in range(1, MAX_PAGES + 1):

                total_now = len(df) + len(repos)

                if total_now >= TARGET:
                    stop_collection = True
                    break

                params = {
                    "q": query,
                    "per_page": PER_PAGE,
                    "page": page
                }

                # ------------------------------------------------
                # REQUEST WITH RATE LIMIT HANDLING
                # ------------------------------------------------

                wait_time = DEFAULT_WAIT

                while True:

                    try:

                        response = requests.get(
                            url,
                            headers=headers,
                            params=params,
                            timeout=30
                        )

                    except requests.RequestException as e:

                        print("Request error:", e)
                        print("Waiting 30 seconds...")
                        time.sleep(30)
                        continue

                    # Successful request
                    if response.status_code == 200:
                        break

                    # Rate limit / secondary rate limit
                    elif response.status_code in [403, 429]:

                        retry_after = response.headers.get(
                            "Retry-After"
                        )

                        if retry_after:
                            wait = int(retry_after)
                        else:
                            wait = wait_time

                        print()
                        print(
                            f"Rate limited. "
                            f"Waiting {wait} seconds..."
                        )

                        time.sleep(wait)

                        wait_time = min(
                            wait_time * 2,
                            600
                        )

                    # 422 = too many results / invalid query
                    elif response.status_code == 422:

                        print(
                            "422: This search partition has "
                            "too many results."
                        )

                        print(
                            "Skipping this partition..."
                        )

                        break

                    else:

                        print(
                            "Error:",
                            response.status_code
                        )

                        try:
                            print(response.json())
                        except:
                            print(response.text)

                        break

                # ------------------------------------------------
                # Stop current query if request failed
                # ------------------------------------------------

                if response.status_code != 200:
                    break

                # ------------------------------------------------
                # READ RESPONSE
                # ------------------------------------------------

                data = response.json()

                items = data.get(
                    "items",
                    []
                )

                if not items:

                    print(
                        "No more repositories "
                        "in this partition."
                    )

                    break

                # ------------------------------------------------
                # ADD NEW REPOSITORIES
                # ------------------------------------------------

                new_count = 0

                for repo in items:

                    repo_id = repo.get("id")

                    if repo_id is None:
                        continue

                    if repo_id not in seen:

                        seen.add(repo_id)

                        repos.append(repo)

                        new_count += 1

                total_now = len(df) + len(repos)

                print(
                    f"Page {page} | "
                    f"New: {new_count} | "
                    f"Total: {total_now}"
                )

                # ------------------------------------------------
                # SAVE EVERY 1000 NEW REPOSITORIES
                # ------------------------------------------------

                if (
                    len(repos) > 0
                    and len(repos) % 1000 < new_count
                ):

                    save_progress()

                # ------------------------------------------------
                # TARGET REACHED
                # ------------------------------------------------

                if total_now >= TARGET:

                    stop_collection = True
                    break

                # ------------------------------------------------
                # DELAY
                # ------------------------------------------------

                time.sleep(REQUEST_DELAY)

            # End pages

        # End date ranges

    # End star ranges

# End languages


# ============================================================
# FINAL DATASET
# ============================================================

print()
print("=======================================")
print("Creating final dataset...")
print("=======================================")

df_final = pd.concat(
    [
        df,
        pd.DataFrame(repos)
    ],
    ignore_index=True
)

# Remove duplicate repositories
df_final = df_final.drop_duplicates(
    subset="id"
).reset_index(drop=True)

# Keep exactly TARGET rows
if len(df_final) > TARGET:
    df_final = df_final.iloc[:TARGET]

# Save final dataset
df_final.to_csv(
    "github_repos_100k_raw.csv",
    index=False
)

# Also update progress file
df_final.to_csv(
    "github_repos_progress.csv",
    index=False
)

print()
print("=======================================")
print("COLLECTION COMPLETE")
print("=======================================")
print("Repositories:", len(df_final))
print("Shape:", df_final.shape)
print("Saved:")
print("  github_repos_100k_raw.csv")
print("  github_repos_progress.csv")
print("=======================================")

Loading existing dataset...


C:\Users\Anandakrishnan VB\AppData\Local\Temp\ipykernel_19608\1001369856.py:79: DtypeWarning: Columns (0: mirror_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("github_repos_100k_raw.csv")


Existing repositories: 53659
Existing shape: (53659, 83)
---------------------------------------
Starting collection
Existing: 53659
Still needed: 46341
---------------------------------------
Date partitions: 34

Searching:
language:Python stars:10..100 created:2010-01-01..2010-06-30
Page 1 | New: 100 | Total: 53759
Page 2 | New: 100 | Total: 53859
Page 3 | New: 100 | Total: 53959
Page 4 | New: 100 | Total: 54059
Page 5 | New: 100 | Total: 54159
Page 6 | New: 100 | Total: 54259
Page 7 | New: 100 | Total: 54359
Page 8 | New: 100 | Total: 54459
Page 9 | New: 100 | Total: 54559
Page 10 | New: 91 | Total: 54650

Searching:
language:Python stars:10..100 created:2010-07-01..2010-12-31
Page 1 | New: 92 | Total: 54742

>>> PROGRESS SAVED
Repositories: 54742

Page 2 | New: 100 | Total: 54842
Page 3 | New: 100 | Total: 54942
Page 4 | New: 100 | Total: 55042
Page 5 | New: 100 | Total: 55142
Page 6 | New: 100 | Total: 55242
Page 7 | New: 100 | Total: 55342
Page 8 | New: 100 | Total: 55442
Page 9 

In [6]:
import pandas as pd

df = pd.read_csv("github_repos_100k_raw.csv")



C:\Users\Anandakrishnan VB\AppData\Local\Temp\ipykernel_19608\1910810997.py:3: DtypeWarning: Columns (0: mirror_url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("github_repos_100k_raw.csv")


In [7]:
df.shape

(100000, 83)

In [8]:
df.head(10)

,id,node_id,name,full_name,private,owner,html_url,description,fork,url,...,has_pull_requests,pull_request_creation_policy,topics,visibility,forks,open_issues,watchers,default_branch,permissions,score
0,1136718169,R_kgDOQ8DxWQ,bmad-assist,Pawel-N-pl/bmad-assist,False,"{'login': 'Pawel-N-pl', 'id': 255624086, 'node...",https://github.com/Pawel-N-pl/bmad-assist,⚠️ Deprecated & unmaintained. This tool only e...,False,https://api.github.com/repos/Pawel-N-pl/bmad-a...,...,True,all,[],public,24,10,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
1,5304158,MDEwOlJlcG9zaXRvcnk1MzA0MTU4,dataconverters,rufuspollock-okfn/dataconverters,False,"{'login': 'rufuspollock-okfn', 'id': 153551630...",https://github.com/rufuspollock-okfn/dataconve...,Python library and command line tool for conve...,False,https://api.github.com/repos/rufuspollock-okfn...,...,True,all,"['convert-data', 'python', 'python-library']",public,31,26,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
2,193336762,MDEwOlJlcG9zaXRvcnkxOTMzMzY3NjI=,Lyapunov-optimization,CrQiu/Lyapunov-optimization,False,"{'login': 'CrQiu', 'id': 34020154, 'node_id': ...",https://github.com/CrQiu/Lyapunov-optimization,Codes for Lyapunov optimization.,False,https://api.github.com/repos/CrQiu/Lyapunov-op...,...,True,all,[],public,22,1,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
3,872866934,R_kgDONAbkdg,CashCoach,Zainabfadeyi/CashCoach,False,"{'login': 'Zainabfadeyi', 'id': 70825458, 'nod...",https://github.com/Zainabfadeyi/CashCoach,NaN,False,https://api.github.com/repos/Zainabfadeyi/Cash...,...,True,all,[],public,1,0,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
4,869732635,R_kgDOM9cRGw,DiffAbXL,AstraZeneca/DiffAbXL,False,"{'login': 'AstraZeneca', 'id': 16338928, 'node...",https://github.com/AstraZeneca/DiffAbXL,The official implementation of DiffAbXL benchm...,False,https://api.github.com/repos/AstraZeneca/DiffAbXL,...,True,all,"['antibody-design', 'binding-affinity', 'diffu...",public,9,0,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
5,23509035,MDEwOlJlcG9zaXRvcnkyMzUwOTAzNQ==,HinetPy,seisman/HinetPy,False,"{'login': 'seisman', 'id': 3974108, 'node_id':...",https://github.com/seisman/HinetPy,A Python package for accessing and processing ...,False,https://api.github.com/repos/seisman/HinetPy,...,True,all,"['hinet', 'python', 'sac', 'seismology', 'win32']",public,42,1,100,main,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
6,232568489,MDEwOlJlcG9zaXRvcnkyMzI1Njg0ODk=,jscat,hanc00l/jscat,False,"{'login': 'hanc00l', 'id': 12759172, 'node_id'...",https://github.com/hanc00l/jscat,JScript RAT,False,https://api.github.com/repos/hanc00l/jscat,...,True,all,[],public,25,0,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
7,266132246,MDEwOlJlcG9zaXRvcnkyNjYxMzIyNDY=,nannernest,EthanRosenthal/nannernest,False,"{'login': 'EthanRosenthal', 'id': 7435500, 'no...",https://github.com/EthanRosenthal/nannernest,Optimal peanut butter and banana sandwiches,False,https://api.github.com/repos/EthanRosenthal/na...,...,True,all,"['computer-vision', 'deep-learning', 'machine-...",public,3,5,100,master,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
8,337459381,MDEwOlJlcG9zaXRvcnkzMzc0NTkzODE=,Deep_Hierarchical_Classification,Ugenteraan/Deep_Hierarchical_Classification,False,"{'login': 'Ugenteraan', 'id': 17710137, 'node_...",https://github.com/Ugenteraan/Deep_Hierarchica...,PyTorch Implementation of Deep Hierarchical Cl...,False,https://api.github.com/repos/Ugenteraan/Deep_H...,...,True,all,"['deep-learning', 'hierarchical-classification...",public,22,3,100,main,"{'admin': False, 'maintain': False, 'push': Fa...",1.0
9,11390366,MDEwOlJlcG9zaXRvcnkxMTM5MDM2Ng==,pypi-notifier,cenkalti/pypi-notifier,False,"{'login': 'cenkalti', 'id': 661618, 'node_id':...",https://github.com/cenkalti/pypi-notifier,📨 A web service that monitors your requirement...,False,https://api.github.com/repos/cenkalti/p